# phenoforge exploration

Exercises the engine directly against `data/vocab.duckdb` — no MCP transport, just the
functions `mcp/server.py` delegates to. Requires `python scripts/load_vocab.py data/athena`
to have been run first.

Exploration only — this notebook is never pushed to production; reusable logic lives in
`src/phenoforge/`.

In [1]:
import os
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's own directory, not the repo
# root, so relative data/ paths below would silently look in notebooks/data/.
# Walk up to the directory containing pyproject.toml and chdir there once.
_dir = Path.cwd()
while not (_dir / "pyproject.toml").exists():
    if _dir.parent == _dir:
        raise FileNotFoundError(f"Could not find repo root (pyproject.toml) above {Path.cwd()}")
    _dir = _dir.parent
os.chdir(_dir)
print(f"cwd: {Path.cwd()}")

from phenoforge.engine.curated import find_matching_cohort, load_curated_concept_set
from phenoforge.engine.db import DEFAULT_VOCAB_DB_PATH, connect
from phenoforge.engine.dense import DenseRetriever
from phenoforge.engine.expansion import expand_descendants
from phenoforge.engine.hybrid import hybrid_search
from phenoforge.engine.lookup import lookup_by_code
from phenoforge.engine.retrieval import BM25Retriever

import json

if not DEFAULT_VOCAB_DB_PATH.exists():
    raise FileNotFoundError(
        f"{DEFAULT_VOCAB_DB_PATH} not found. Run: python scripts/load_vocab.py data/athena"
    )

con = connect()
print(f"Connected to {DEFAULT_VOCAB_DB_PATH}")

cwd: /Users/colbywilkinson/projects/phenoforge
Connected to data/vocab.duckdb


## `lookup_concept` — exact code lookup

In [2]:
lookup_by_code(con, "E11.21")

Concept(concept_id=45591027, concept_code='E11.21', concept_name='Type 2 diabetes mellitus with diabetic nephropathy', domain_id='Condition', vocabulary_id='ICD10CM')

## `expand_hierarchy` — descendant expansion

In [3]:
descendants = expand_descendants(con, "E11")
for c in descendants:
    print(c.concept_code, "-", c.concept_name, f"[{c.source}]")

E11.52 - Type 2 diabetes mellitus with diabetic peripheral angiopathy with gangrene [hierarchy_expansion:E11]
E11.618 - Type 2 diabetes mellitus with other diabetic arthropathy [hierarchy_expansion:E11]
E11.3299 - Type 2 diabetes mellitus with mild nonproliferative diabetic retinopathy without macular edema, unspecified eye [hierarchy_expansion:E11]
E11.3313 - Type 2 diabetes mellitus with moderate nonproliferative diabetic retinopathy with macular edema, bilateral [hierarchy_expansion:E11]
E11.3493 - Type 2 diabetes mellitus with severe nonproliferative diabetic retinopathy without macular edema, bilateral [hierarchy_expansion:E11]
E11.3512 - Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, left eye [hierarchy_expansion:E11]
E11.3519 - Type 2 diabetes mellitus with proliferative diabetic retinopathy with macular edema, unspecified eye [hierarchy_expansion:E11]
E11.37X2 - Type 2 diabetes mellitus with diabetic macular edema, resolved following treatm

## `search_concepts` — hybrid BM25 + dense

Builds the BM25 index and, if `data/concept_index.lance` exists, loads the real
BioLORD-2023 model — expect a pause on first run.

In [4]:
bm25 = BM25Retriever(con)

index_path = DEFAULT_VOCAB_DB_PATH.parent / "concept_index.lance"
dense = DenseRetriever(con, index_path=index_path) if index_path.exists() else None
print("Dense index:", "loaded" if dense else "not built — BM25-only fallback")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense index: loaded


In [6]:
results, unmappable = hybrid_search(bm25, dense, "diabetes with kidney damage", k=10)
if unmappable:
    print("Unmappable:", unmappable.reason)
else:
    for c in results:
        print(c.concept_code, "-", c.concept_name, f"[{c.source}]")

E13.2 - Other specified diabetes mellitus with kidney complications [dense:diabetes with kidney damage]
E11.2 - Type 2 diabetes mellitus with kidney complications [bm25:diabetes with kidney damage]
E13.22 - Other specified diabetes mellitus with diabetic chronic kidney disease [dense:diabetes with kidney damage]
E11.29 - Type 2 diabetes mellitus with other diabetic kidney complication [bm25:diabetes with kidney damage]
E13.29 - Other specified diabetes mellitus with other diabetic kidney complication [dense:diabetes with kidney damage]
E10.2 - Type 1 diabetes mellitus with kidney complications [bm25:diabetes with kidney damage]
E08.2 - Diabetes mellitus due to underlying condition with kidney complications [dense:diabetes with kidney damage]
E13.21 - Other specified diabetes mellitus with diabetic nephropathy [dense:diabetes with kidney damage]
E09.2 - Drug or chemical induced diabetes mellitus with kidney complications [bm25:diabetes with kidney damage]
E08.21 - Diabetes mellitus due 

## `find_curated_definition` — OHDSI Phenotype Library match

In [8]:
library_dir = Path("data/phenotype_library")
manifest_path = library_dir / "manifest.json"

query = "diabetic kidney disease"

if not manifest_path.exists():
    print("No phenotype library — run: python scripts/fetch_phenotype_library.py")
else:
    manifest = json.loads(manifest_path.read_text())
    cohort_id = find_matching_cohort(query, manifest)
    if cohort_id is None:
        print("No bundled cohort matches (or the match was ambiguous).")
    else:
        concept_set = load_curated_concept_set(con, cohort_id, library_dir)
        print(f"Matched cohort {cohort_id}: {manifest[cohort_id]}")
        for c in concept_set.concepts:
            print(" ", c.concept_code, "-", c.concept_name)

Matched cohort 687: Chronic kidney disease
  N18.9 - Chronic kidney disease, unspecified
  N18 - Chronic kidney disease (CKD)
  E08.22 - Diabetes mellitus due to underlying condition with diabetic chronic kidney disease
